# Objetivo 4 v2 — Entrenamiento GPU (Corregido segun PPI)
**Correcciones vs v1:**
- HQC-CNN: encoding RY + entrelazamiento circular
- HQCINN-deep: 6 qubits (Akpinar et al.)
- ResNet-18: fc = Linear(512, n_classes) directo
- batch=32 para ResNet-18, batch=16 para HQCNN
- Class weighting activado en AMBOS datasets
- IC 95% via 5-fold CV estratificado (segun PPI)

**Resultados v1 (referencia):**
- Chest: todos los modelos F1>0.91, sin diferencias significativas (ANOVA p>>0.05)
- Lung: HQCNN colapsaron (F1~0.23) por batch=64 + encoding incorrecto. ResNet-18 F1=0.95

## 0 · Instalacion

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','-q','torch','torchvision','--index-url','https://download.pytorch.org/whl/cu128'],check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','pennylane>=0.38','pennylane-lightning','scikit-learn','pandas','matplotlib','seaborn','tqdm'],check=False)
import torch; print('PyTorch:',torch.__version__)
print('CUDA:',torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))


## 1 · Imports y seed

In [ ]:
import os,random,time,json
from pathlib import Path
from itertools import combinations
import numpy as np,pandas as pd
import matplotlib.pyplot as plt,seaborn as sns
import torch,torch.nn as nn
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18,ResNet18_Weights
from torch.utils.data import DataLoader,WeightedRandomSampler
import pennylane as qml
from pennylane.qnn import TorchLayer
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score,f1_score,classification_report,confusion_matrix,roc_auc_score,recall_score
from sklearn.model_selection import StratifiedKFold
from scipy import stats
SEED=42
random.seed(SEED);np.random.seed(SEED)
torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark=True
torch.backends.cudnn.deterministic=False
torch.backends.cuda.matmul.allow_tf32=True
torch.backends.cudnn.allow_tf32=True
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type=='cuda','GPU no detectada'
print('Device:',DEVICE,'|',torch.cuda.get_device_name(0))


## 2 · Rutas y transforms

In [ ]:
BASE_DIR=Path(os.getcwd())
CHEST_OUT=BASE_DIR/'etl_output'/'chest_xray'
LUNG_OUT=BASE_DIR/'etl_output'/'lung_cancer'
CKPT_DIR=BASE_DIR/'checkpoints_gpu_v2';CKPT_DIR.mkdir(exist_ok=True)
IMG_SIZE=(128,128);MEAN=[0.485,0.456,0.406];STD=[0.229,0.224,0.225]
tfm_tr=T.Compose([T.Resize(IMG_SIZE),T.Grayscale(num_output_channels=3),T.RandomRotation(10),T.RandomHorizontalFlip(),T.ColorJitter(brightness=0.15),T.RandomAffine(degrees=0,scale=(0.90,1.10)),T.ToTensor(),T.Normalize(MEAN,STD)])
tfm_ev=T.Compose([T.Resize(IMG_SIZE),T.Grayscale(num_output_channels=3),T.ToTensor(),T.Normalize(MEAN,STD)])
N_CHEST=2;N_LUNG=3
CHEST_NAMES=['NORMAL','PNEUMONIA'];LUNG_NAMES=['Benign','Malignant','Normal']


## 3 · Class weights (ambos datasets segun PPI)

In [ ]:
def get_class_weights(root):
    ds=ImageFolder(Path(root)/'train')
    tgts=torch.tensor(ds.targets)
    cw=(1.0/torch.bincount(tgts).float())
    return (cw/cw.sum()).to(DEVICE)

chest_cw=get_class_weights(CHEST_OUT)
lung_cw=get_class_weights(LUNG_OUT)
print('Chest class weights:',chest_cw)
print('Lung  class weights:',lung_cw)


## 4 · Modelos (corregidos)

In [ ]:
N_Q_SHALLOW=4; N_Q_DEEP=6

def get_qdev(n):
    try: return qml.device('lightning.qubit',wires=n)
    except: return qml.device('default.qubit',wires=n)

# ResNet-18: fc = Linear(512, n_classes) segun PPI
def build_resnet18(nc):
    m=resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    m.fc=nn.Linear(m.fc.in_features,nc)
    return m

# HQC-CNN: encoding RY + entrelazamiento circular (Dong et al., 2023)
dev_hqccnn=get_qdev(4)
@qml.qnode(dev_hqccnn,interface='torch',diff_method='adjoint')
def circuit_hqccnn(inputs,weights_rx,weights_ry):
    qml.AngleEmbedding(inputs,wires=range(4),rotation='Y')
    for q in range(4): qml.RX(weights_rx[q],wires=q); qml.RY(weights_ry[q],wires=q)
    for q in range(4): qml.CNOT(wires=[q,(q+1)%4])
    return [qml.expval(qml.PauliZ(i)) for i in range(4)]

class HQCCNN(nn.Module):
    def __init__(self,nc):
        super().__init__()
        self.feat=nn.Sequential(nn.Conv2d(3,16,3,padding=1),nn.BatchNorm2d(16),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(16,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.AdaptiveAvgPool2d((4,4)))
        self.pre_q=nn.Sequential(nn.Linear(64*4*4,64),nn.ReLU(),nn.Linear(64,4),nn.Tanh())
        self.ql=TorchLayer(circuit_hqccnn,{'weights_rx':(4,),'weights_ry':(4,)})
        self.cls=nn.Linear(4,nc)
    def forward(self,x):
        dv=x.device; x=self.feat(x).flatten(1)
        qi=self.pre_q(x).float().cpu()*torch.pi
        return self.cls(self.ql(qi).to(dv).to(x.dtype))

# PEQML: BasicEntanglerLayers + RY (Abdur & Kim, 2025)
dev_peqml=get_qdev(4)
@qml.qnode(dev_peqml,interface='torch',diff_method='adjoint')
def circuit_peqml(inputs,weights):
    qml.AngleEmbedding(inputs,wires=range(4),rotation='Y')
    qml.BasicEntanglerLayers(weights,wires=range(4))
    return [qml.expval(qml.PauliZ(i)) for i in range(4)]

class PEQML(nn.Module):
    def __init__(self,nc):
        super().__init__()
        def dw(ic,oc,s=1): return nn.Sequential(nn.Conv2d(ic,ic,3,stride=s,padding=1,groups=ic,bias=False),nn.Conv2d(ic,oc,1,bias=False),nn.BatchNorm2d(oc),nn.ReLU6())
        self.feat=nn.Sequential(nn.Conv2d(3,8,3,stride=2,padding=1,bias=False),nn.BatchNorm2d(8),nn.ReLU6(),dw(8,16,2),dw(16,32,2),dw(32,32,2),nn.AdaptiveAvgPool2d((2,2)))
        self.pre_q=nn.Sequential(nn.Linear(128,4),nn.Tanh())
        self.ql=TorchLayer(circuit_peqml,{'weights':(2,4)})
        self.cls=nn.Linear(4,nc)
    def forward(self,x):
        dv=x.device; x=self.feat(x).flatten(1)
        qi=self.pre_q(x).float().cpu()*torch.pi
        return self.cls(self.ql(qi).to(dv).to(x.dtype))

# HQCINN: shallow=4q/1capa  deep=6q/3capas (Akpinar et al., 2025)
def build_hqcinn_qlayer(nq,nl,ent='linear'):
    dev=get_qdev(nq)
    def entangle(wires,et):
        n=len(wires)
        if et=='linear':    [qml.CNOT(wires=[wires[i],wires[i+1]]) for i in range(n-1)]
        elif et=='circular':[qml.CNOT(wires=[wires[i],wires[(i+1)%n]]) for i in range(n)]
        elif et=='full':    [qml.CNOT(wires=[wires[i],wires[j]]) for i in range(n) for j in range(i+1,n)]
    @qml.qnode(dev,interface='torch',diff_method='adjoint')
    def circuit(inputs,weights_rx,weights_ry,weights_rz):
        qml.AngleEmbedding(inputs,wires=range(nq),rotation='X')
        for l in range(nl):
            for q in range(nq): qml.RX(weights_rx[l,q],wires=q); qml.RY(weights_ry[l,q],wires=q); qml.RZ(weights_rz[l,q],wires=q)
            entangle(list(range(nq)),ent)
        return [qml.expval(qml.PauliZ(i)) for i in range(nq)]
    ws={k:(nl,nq) for k in ['weights_rx','weights_ry','weights_rz']}
    return TorchLayer(circuit,ws)

class HQCINN(nn.Module):
    def __init__(self,nc,variant='shallow',entanglement='linear'):
        super().__init__()
        nq,nl={'shallow':(N_Q_SHALLOW,1),'deep':(N_Q_DEEP,3)}[variant]
        self.nq=nq
        self.feat=nn.Sequential(nn.Conv2d(3,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(),nn.AdaptiveAvgPool2d((2,2)))
        self.pre_q=nn.Sequential(nn.Linear(128*2*2,nq),nn.Tanh())
        self.ql=build_hqcinn_qlayer(nq,nl,entanglement)
        self.cls=nn.Linear(nq,nc)
    def forward(self,x):
        dv=x.device; x=self.feat(x).flatten(1)
        qi=self.pre_q(x).float().cpu()*torch.pi
        return self.cls(self.ql(qi).to(dv).to(x.dtype))

def build_model(name,nc):
    if name=='resnet18': return build_resnet18(nc)
    elif name=='hqccnn': return HQCCNN(nc)
    elif name=='peqml': return PEQML(nc)
    elif name=='hqcinn_shallow': return HQCINN(nc,variant='shallow')
    elif name=='hqcinn_deep': return HQCINN(nc,variant='deep')
    else: raise ValueError(name)

MODEL_NAMES=['resnet18','hqccnn','peqml','hqcinn_shallow','hqcinn_deep']
print('Modelos listos. Qubits: HQC-CNN=4, PEQML=4, HQCINN-shallow=4, HQCINN-deep=6')


## 5 · Funciones de entrenamiento (corregidas)

In [ ]:
# batch=32 para ResNet-18, batch=16 para HQCNN (segun PPI)
BS_CLASSICAL=32; BS_QUANTUM=16

def make_loaders(root,weighted=False,bs=32):
    root=Path(root)
    ds_tr=ImageFolder(root/'train',transform=tfm_tr)
    ds_va=ImageFolder(root/'val',transform=tfm_ev)
    ds_te=ImageFolder(root/'test',transform=tfm_ev)
    kw=dict(num_workers=4,pin_memory=True,persistent_workers=True)
    if weighted:
        tgts=torch.tensor(ds_tr.targets)
        sw=(1.0/torch.bincount(tgts).float())[tgts]
        ltr=DataLoader(ds_tr,batch_size=bs,sampler=WeightedRandomSampler(sw,len(sw),True),**kw)
    else:
        ltr=DataLoader(ds_tr,batch_size=bs,shuffle=True,**kw)
    lva=DataLoader(ds_va,batch_size=bs,shuffle=False,**kw)
    lte=DataLoader(ds_te,batch_size=bs,shuffle=False,**kw)
    return ltr,lva,lte,ds_tr.class_to_idx

# class weights para AMBOS datasets (segun PPI)
def get_class_weights(root):
    ds=ImageFolder(Path(root)/'train')
    tgts=torch.tensor(ds.targets)
    cw=(1.0/torch.bincount(tgts).float())
    return (cw/cw.sum()).to(DEVICE)

def train_one(model,tr_l,va_l,cw=None,max_ep=50,patience=5,ckpt=None):
    model=model.to(DEVICE)
    crit=nn.CrossEntropyLoss(weight=cw)
    opt=torch.optim.Adam(model.parameters(),lr=1e-3)
    sch=torch.optim.lr_scheduler.MultiStepLR(opt,milestones=[10],gamma=0.1)
    scaler=torch.amp.GradScaler('cuda')
    hist={'train_loss':[],'val_loss':[],'val_acc':[],'val_f1':[]}
    best_f1,no_imp=0.0,0
    for ep in range(1,max_ep+1):
        model.train();tloss=0.0;t0=time.time()
        for imgs,lbls in tqdm(tr_l,desc='Ep '+str(ep),leave=False):
            imgs,lbls=imgs.to(DEVICE,non_blocking=True),lbls.to(DEVICE,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'): logits=model(imgs)
            loss=crit(logits.float(),lbls)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tloss+=loss.item()*imgs.size(0)
        sch.step(); avg_tr=tloss/len(tr_l.dataset); ep_t=time.time()-t0
        model.eval();vloss,preds,labs=0.0,[],[]
        with torch.no_grad():
            for imgs,lbls in va_l:
                imgs,lbls=imgs.to(DEVICE,non_blocking=True),lbls.to(DEVICE,non_blocking=True)
                with torch.amp.autocast('cuda'): logits=model(imgs)
                vloss+=crit(logits.float(),lbls).item()*imgs.size(0)
                preds.extend(logits.argmax(1).cpu().numpy()); labs.extend(lbls.cpu().numpy())
        avg_va=vloss/len(va_l.dataset)
        acc=sum(p==l for p,l in zip(preds,labs))/len(labs)
        f1=f1_score(labs,preds,average='macro',zero_division=0)
        hist['train_loss'].append(avg_tr);hist['val_loss'].append(avg_va)
        hist['val_acc'].append(acc);hist['val_f1'].append(f1)
        print('Ep {:3d} | {:.1f}s | tr={:.4f} | va={:.4f} | acc={:.4f} | F1={:.4f}'.format(ep,ep_t,avg_tr,avg_va,acc,f1))
        if f1>best_f1:
            best_f1,no_imp=f1,0
            if ckpt: torch.save(model.state_dict(),ckpt)
        else:
            no_imp+=1
            if no_imp>=patience: print('Early stopping ep',ep); break
    return hist

def compute_metrics(model,te_l,cls_names,mname):
    model=model.to(DEVICE).eval()
    preds,labs,probs=[],[],[]
    t0=time.time()
    with torch.no_grad():
        for imgs,lbls in te_l:
            with torch.amp.autocast('cuda'): logits=model(imgs.to(DEVICE,non_blocking=True))
            probs.extend(torch.softmax(logits.float(),1).cpu().numpy())
            preds.extend(logits.argmax(1).cpu().numpy()); labs.extend(lbls.numpy())
    inf_ms=(time.time()-t0)/len(labs)*1000
    nc=len(cls_names); acc=accuracy_score(labs,preds)
    f1m=f1_score(labs,preds,average='macro',zero_division=0)
    sens=recall_score(labs,preds,average='macro',zero_division=0)
    cm=confusion_matrix(labs,preds)
    sl=[]
    for i in range(nc):
        TP=cm[i,i];FN=cm[i,:].sum()-TP;FP=cm[:,i].sum()-TP;TN=cm.sum()-TP-FN-FP
        sl.append(TN/(TN+FP) if (TN+FP)>0 else 0.0)
    spec=float(np.mean(sl))
    try: auc=roc_auc_score(labs,probs,multi_class='ovr',average='macro') if nc>2 else roc_auc_score(labs,[p[1] for p in probs])
    except: auc=float('nan')
    params=sum(p.numel() for p in model.parameters() if p.requires_grad)
    print('\n===',mname,'===')
    print('  Acc={:.4f}  F1={:.4f}  Sens={:.4f}  Spec={:.4f}  AUC={:.4f}'.format(acc,f1m,sens,spec,auc))
    print('  Inf={:.3f}ms  Params={:,}'.format(inf_ms,params))
    print(classification_report(labs,preds,target_names=cls_names,zero_division=0))
    cm2=confusion_matrix(labs,preds)
    plt.figure(figsize=(max(4,nc*2),max(3,nc*1.5)))
    sns.heatmap(cm2,annot=True,fmt='d',cmap='Blues',xticklabels=cls_names,yticklabels=cls_names)
    plt.title('Confusion Matrix - '+mname);plt.ylabel('Real');plt.xlabel('Predicho')
    plt.tight_layout();plt.show()
    return dict(model=mname,acc=acc,f1=f1m,sens=sens,spec=spec,auc=auc,inf_ms=inf_ms,params=params,preds=preds,labs=labs,probs=probs)

def plot_history(hist,title):
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    fig.suptitle('Curvas - '+title,fontweight='bold')
    axes[0].plot(hist['train_loss'],label='train');axes[0].plot(hist['val_loss'],label='val')
    axes[0].set_title('Loss');axes[0].legend()
    axes[1].plot(hist['val_acc'],label='acc');axes[1].plot(hist['val_f1'],label='F1-macro')
    axes[1].set_title('Val/F1');axes[1].legend()
    plt.tight_layout();plt.show()

print('Funciones listas. Correciones aplicadas: batch=32/16, class_weights en chest+lung, fc=512->n_classes')


## 6 · Inicializar resultados y checkpoints

In [ ]:
results_chest={};results_lung={}
ckpt_chest=CKPT_DIR/'chest';ckpt_chest.mkdir(exist_ok=True)
ckpt_lung=CKPT_DIR/'lung';ckpt_lung.mkdir(exist_ok=True)
print('Checkpoints en:',CKPT_DIR)


## DATASET 1: Chest X-Ray (Pneumonia)
> batch=32 para ResNet-18, batch=16 para HQCNN

### ResNet-18 — Chest X-Ray (batch=32)

In [ ]:
_mn='resnet18'
_bs=32
_ckpt=str(ckpt_chest/('resnet18_best.pt'))
_tr,_va,_te,_=make_loaders(CHEST_OUT,weighted=True,bs=_bs)
_model=build_model(_mn,N_CHEST).to(DEVICE)
_hist=train_one(_model,_tr,_va,cw=chest_cw,max_ep=50,patience=5,ckpt=_ckpt)
plot_history(_hist,'resnet18-chest')
_model.load_state_dict(torch.load(_ckpt,map_location=DEVICE))
results_chest['resnet18']=compute_metrics(_model,_te,CHEST_NAMES,'resnet18/chest')
del _model;torch.cuda.empty_cache()


### HQC-CNN — Chest X-Ray (batch=16)

In [ ]:
_mn='hqccnn'
_bs=16
_ckpt=str(ckpt_chest/('hqccnn_best.pt'))
_tr,_va,_te,_=make_loaders(CHEST_OUT,weighted=True,bs=_bs)
_model=build_model(_mn,N_CHEST).to(DEVICE)
_hist=train_one(_model,_tr,_va,cw=chest_cw,max_ep=50,patience=5,ckpt=_ckpt)
plot_history(_hist,'hqccnn-chest')
_model.load_state_dict(torch.load(_ckpt,map_location=DEVICE))
results_chest['hqccnn']=compute_metrics(_model,_te,CHEST_NAMES,'hqccnn/chest')
del _model;torch.cuda.empty_cache()


### PEQML — Chest X-Ray (batch=16)

In [ ]:
_mn='peqml'
_bs=16
_ckpt=str(ckpt_chest/('peqml_best.pt'))
_tr,_va,_te,_=make_loaders(CHEST_OUT,weighted=True,bs=_bs)
_model=build_model(_mn,N_CHEST).to(DEVICE)
_hist=train_one(_model,_tr,_va,cw=chest_cw,max_ep=50,patience=5,ckpt=_ckpt)
plot_history(_hist,'peqml-chest')
_model.load_state_dict(torch.load(_ckpt,map_location=DEVICE))
results_chest['peqml']=compute_metrics(_model,_te,CHEST_NAMES,'peqml/chest')
del _model;torch.cuda.empty_cache()


### HQCINN-shallow — Chest X-Ray (batch=16)

In [ ]:
_mn='hqcinn_shallow'
_bs=16
_ckpt=str(ckpt_chest/('hqcinn_shallow_best.pt'))
_tr,_va,_te,_=make_loaders(CHEST_OUT,weighted=True,bs=_bs)
_model=build_model(_mn,N_CHEST).to(DEVICE)
_hist=train_one(_model,_tr,_va,cw=chest_cw,max_ep=50,patience=5,ckpt=_ckpt)
plot_history(_hist,'hqcinn_shallow-chest')
_model.load_state_dict(torch.load(_ckpt,map_location=DEVICE))
results_chest['hqcinn_shallow']=compute_metrics(_model,_te,CHEST_NAMES,'hqcinn_shallow/chest')
del _model;torch.cuda.empty_cache()


### HQCINN-deep — Chest X-Ray (batch=16)

In [ ]:
_mn='hqcinn_deep'
_bs=16
_ckpt=str(ckpt_chest/('hqcinn_deep_best.pt'))
_tr,_va,_te,_=make_loaders(CHEST_OUT,weighted=True,bs=_bs)
_model=build_model(_mn,N_CHEST).to(DEVICE)
_hist=train_one(_model,_tr,_va,cw=chest_cw,max_ep=50,patience=5,ckpt=_ckpt)
plot_history(_hist,'hqcinn_deep-chest')
_model.load_state_dict(torch.load(_ckpt,map_location=DEVICE))
results_chest['hqcinn_deep']=compute_metrics(_model,_te,CHEST_NAMES,'hqcinn_deep/chest')
del _model;torch.cuda.empty_cache()


## DATASET 2: IQ-OTH/NCCD Lung Cancer
> batch=32 para ResNet-18, batch=16 para HQCNN

### ResNet-18 — Lung Cancer (batch=32)

In [ ]:
_mn='resnet18'
_bs=32
_ckpt=str(ckpt_lung/('resnet18_best.pt'))
_tr,_va,_te,_=make_loaders(LUNG_OUT,weighted=True,bs=_bs)
_model=build_model(_mn,N_LUNG).to(DEVICE)
_hist=train_one(_model,_tr,_va,cw=lung_cw,max_ep=50,patience=5,ckpt=_ckpt)
plot_history(_hist,'resnet18-lung')
_model.load_state_dict(torch.load(_ckpt,map_location=DEVICE))
results_lung['resnet18']=compute_metrics(_model,_te,LUNG_NAMES,'resnet18/lung')
del _model;torch.cuda.empty_cache()


### HQC-CNN — Lung Cancer (batch=16)

In [ ]:
_mn='hqccnn'
_bs=16
_ckpt=str(ckpt_lung/('hqccnn_best.pt'))
_tr,_va,_te,_=make_loaders(LUNG_OUT,weighted=True,bs=_bs)
_model=build_model(_mn,N_LUNG).to(DEVICE)
_hist=train_one(_model,_tr,_va,cw=lung_cw,max_ep=50,patience=5,ckpt=_ckpt)
plot_history(_hist,'hqccnn-lung')
_model.load_state_dict(torch.load(_ckpt,map_location=DEVICE))
results_lung['hqccnn']=compute_metrics(_model,_te,LUNG_NAMES,'hqccnn/lung')
del _model;torch.cuda.empty_cache()


### PEQML — Lung Cancer (batch=16)

In [ ]:
_mn='peqml'
_bs=16
_ckpt=str(ckpt_lung/('peqml_best.pt'))
_tr,_va,_te,_=make_loaders(LUNG_OUT,weighted=True,bs=_bs)
_model=build_model(_mn,N_LUNG).to(DEVICE)
_hist=train_one(_model,_tr,_va,cw=lung_cw,max_ep=50,patience=5,ckpt=_ckpt)
plot_history(_hist,'peqml-lung')
_model.load_state_dict(torch.load(_ckpt,map_location=DEVICE))
results_lung['peqml']=compute_metrics(_model,_te,LUNG_NAMES,'peqml/lung')
del _model;torch.cuda.empty_cache()


### HQCINN-shallow — Lung Cancer (batch=16)

In [ ]:
_mn='hqcinn_shallow'
_bs=16
_ckpt=str(ckpt_lung/('hqcinn_shallow_best.pt'))
_tr,_va,_te,_=make_loaders(LUNG_OUT,weighted=True,bs=_bs)
_model=build_model(_mn,N_LUNG).to(DEVICE)
_hist=train_one(_model,_tr,_va,cw=lung_cw,max_ep=50,patience=5,ckpt=_ckpt)
plot_history(_hist,'hqcinn_shallow-lung')
_model.load_state_dict(torch.load(_ckpt,map_location=DEVICE))
results_lung['hqcinn_shallow']=compute_metrics(_model,_te,LUNG_NAMES,'hqcinn_shallow/lung')
del _model;torch.cuda.empty_cache()


### HQCINN-deep — Lung Cancer (batch=16)

In [ ]:
_mn='hqcinn_deep'
_bs=16
_ckpt=str(ckpt_lung/('hqcinn_deep_best.pt'))
_tr,_va,_te,_=make_loaders(LUNG_OUT,weighted=True,bs=_bs)
_model=build_model(_mn,N_LUNG).to(DEVICE)
_hist=train_one(_model,_tr,_va,cw=lung_cw,max_ep=50,patience=5,ckpt=_ckpt)
plot_history(_hist,'hqcinn_deep-lung')
_model.load_state_dict(torch.load(_ckpt,map_location=DEVICE))
results_lung['hqcinn_deep']=compute_metrics(_model,_te,LUNG_NAMES,'hqcinn_deep/lung')
del _model;torch.cuda.empty_cache()


## 7 · Tablas comparativas

In [ ]:
all_res={'chest':results_chest,'lung':results_lung}
for dsname in ['chest','lung']:
    rows=[{'Modelo':mn,'Accuracy':round(r['acc'],4),'F1-macro':round(r['f1'],4),'Sensibilidad':round(r['sens'],4),'Especificidad':round(r['spec'],4),'AUC':round(r['auc'],4),'Inf(ms)':round(r['inf_ms'],3),'Params':r['params']} for mn,r in all_res[dsname].items()]
    df=pd.DataFrame(rows).set_index('Modelo')
    print('\n===',dsname.upper(),'===');print(df.to_string())
    df.to_csv(BASE_DIR/('resultados_v2_'+dsname+'.csv'))
    mets=['Accuracy','F1-macro','Sensibilidad','Especificidad','AUC']
    fig,axes=plt.subplots(1,len(mets),figsize=(18,4));fig.suptitle('Comparativa v2 - '+dsname,fontweight='bold')
    pal=sns.color_palette('Set2',len(df))
    for ax,met in zip(axes,mets):
        vals=df[met].values;bars=ax.bar(df.index,vals,color=pal)
        ax.set_title(met);ax.set_ylim(0,1.05);ax.tick_params(axis='x',rotation=30,labelsize=8)
        for bar,v in zip(bars,vals): ax.text(bar.get_x()+bar.get_width()/2,v+0.01,'{:.3f}'.format(v),ha='center',fontsize=7)
    plt.tight_layout();plt.show()


## 8 · Analisis estadistico — ANOVA + t-Student + IC 95% (5-fold CV segun PPI)

In [ ]:
# 5-fold CV estratificado sobre test para IC 95% (segun PPI)
from sklearn.model_selection import StratifiedKFold

def cv_f1_ci(preds,labs,n_splits=5,seed=42):
    skf=StratifiedKFold(n_splits=n_splits,shuffle=True,random_state=seed)
    idx=np.arange(len(labs))
    scores=[f1_score(np.array(labs)[te],np.array(preds)[te],average='macro',zero_division=0)
            for _,te in skf.split(idx,labs)]
    return float(np.mean(scores)),np.percentile(scores,[2.5,97.5])

all_res={'chest':results_chest,'lung':results_lung}
for dsname in ['chest','lung']:
    print('\n===',dsname.upper(),'===')
    dists={}
    for mn,r in all_res[dsname].items():
        mean_f1,ci=cv_f1_ci(r['preds'],r['labs'])
        dists[mn]=[f1_score(np.array(r['labs'])[te],np.array(r['preds'])[te],average='macro',zero_division=0)
                   for _,te in StratifiedKFold(5,shuffle=True,random_state=42).split(np.arange(len(r['labs'])),r['labs'])]
        print('  {:<20} F1={:.4f}  IC95=[{:.4f},{:.4f}]'.format(mn,r['f1'],ci[0],ci[1]))
    f_stat,p_anova=stats.f_oneway(*dists.values())
    print('  ANOVA F={:.4f} p={:.6f} sig={}'.format(f_stat,p_anova,p_anova<0.05))
    pair_rows=[]
    for m1,m2 in combinations(list(dists.keys()),2):
        t,p=stats.ttest_ind(dists[m1],dists[m2])
        sig='*' if p<0.05 else ''
        pair_rows.append({'Par':m1+' vs '+m2,'t':round(t,4),'p':round(p,6),'sig':sig})
        print('  {} vs {}: t={:.4f} p={:.6f} {}'.format(m1,m2,t,p,sig))
    pd.DataFrame(pair_rows).to_csv(BASE_DIR/('ttest_v2_'+dsname+'.csv'),index=False)


## 9 · Guardar JSON consolidado

In [ ]:
summary={}
for ds in ['chest','lung']:
    summary[ds]={mn:{k:v for k,v in r.items() if k not in ('preds','labs','probs')} for mn,r in all_res[ds].items()}
out=BASE_DIR/'resultados_v2.json'
with open(out,'w') as f: json.dump(summary,f,indent=2,default=str)
print('Guardado:',out)
